# Partition reanalysis uncertainty between impact model training and GCM post-processing

Adapting code from 3_project_impacts.ipynb to allow cross-projection between different impact model training and bias-correcting/downscaling reanalysis sources - but only keeping the 'main' specification for simplicity

In [14]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_readregs import coeffs_mortality, coeffs_gdppc, coeffs_payroll
dir_list = get_params()

In [2]:
from distributed import Client

# Start dask client 
# (playing around with workers, threads for efficiency)
client = Client(n_workers=6,
    threads_per_worker=3,
    processes=True,
    memory_limit="8GB")
display(client)

/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45509 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/45509/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/45509/status,Workers: 6
Total threads: 18,Total memory: 44.70 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45625,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/45509/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:43521,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/44913/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:36701,


In [3]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':False},
               'gdppc':{'varname':'tassumpoly','geo_typ':'adm1','popwght':True},
               'payroll':{'varname':'days90plus','geo_typ':'adm2-us','popwght':True}}
#params_proc = {'payroll':{'varname':'days90plus','geo_typ':'adm2-us','popwght':True}}
#{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}

In [4]:
coeffs_mortality(return_raw=True)

(<xarray.Dataset> Size: 1kB
 Dimensions:     (proj_base: 4, pred: 4, resolution: 2, timeframe: 2,
                  obstreat: 1, spec: 1, indflag: 2)
 Coordinates:
   * proj_base   (proj_base) <U6 96B 'ERA5' 'GMFD' 'JRA-3Q' 'MERRA2'
   * pred        (pred) object 32B 'tassumpoly_1' ... 'tassumpoly_4'
   * resolution  (resolution) object 16B '025deg' '1deg'
   * timeframe   (timeframe) object 16B '1980-2010' 'ALLYEARS'
   * obstreat    (obstreat) object 8B 'sep'
   * spec        (spec) object 8B 'pooled'
   * indflag     (indflag) object 16B 'noInd' 'wInd'
 Data variables:
     coeffs      (resolution, timeframe, obstreat, spec, indflag, pred, proj_base) float64 1kB ...,
 <xarray.Dataset> Size: 4kB
 Dimensions:     (pred_1: 4, pred_2: 4, proj_base: 4, resolution: 2,
                  timeframe: 2, obstreat: 1, spec: 1, indflag: 2)
 Coordinates:
   * pred_1      (pred_1) object 32B 'tassumpoly_1' ... 'tassumpoly_4'
   * pred_2      (pred_2) object 32B 'tassumpoly_1' ... 'tassumpoly_4'
  

In [15]:
overwrite = True

for proc in params_proc:
    # Get filenames of aggregated climate predictors
    fns = np.sort(glob.glob(f'{dir_list['proc']}aggs/{params_proc[proc]['varname']}_*_{params_proc[proc]['geo_typ']}*'))
    if params_proc[proc]['popwght']:
        fns = [fn for fn in fns if re.search(r'\_popwght\.zarr',fn)]
    else:
        fns = [fn for fn in fns if not re.search(r'\_popwght\.zarr',fn)]
    fns = {re.split(r'\_',re.split(r'\/',fn)[-1])[2]:fn for fn in fns}
    
    # Load regression coefficients
    if proc == 'gdppc':
        coeffs,vcv = coeffs_gdppc(return_raw=False)
    elif proc == 'mortality':
        coeffs,vcv = coeffs_mortality(return_raw=True)
        # Keep the Ind flag to test something
        sub_params = {'resolution':'025deg','timeframe':'1980-2010','obstreat':'sep','spec':'pooled'}
        coeffs = coeffs.sel(sub_params)
        vcv = vcv.sel(sub_params)
    elif proc == 'payroll':
        coeffs,vcv = coeffs_payroll(return_raw=False)
    else:
        raise KeyError()
        
    # Rename proj_bases to allow for cross-processing
    coeffs = coeffs.rename({'proj_base':'fit_mod'})
    
    for mod in tqdm(fns):
        output_fn = f'{dir_list['proc']}impact_projs/{proc}/{re.sub(params_proc[proc]['varname'],f'd{proc}-crossfit',re.split(r'\/',fns[mod])[-1])}'
        
        if overwrite or (not os.path.exists(output_fn)):
            
            # Load aggregated climate predictor
            ds = xr.open_zarr(fns[mod],consolidated=False,decode_timedelta=False)
        
            # Isolate predictor variables
            if params_proc[proc]['varname'] == 'tassumpoly':
                ds['degree'] = [f'tassumpoly_{k}' for k in ds.degree.values]
                ds = ds.rename({'degree':'pred'})
            elif params_proc[proc]['varname'] == 'tasmaxbinF':
                ds['binF'] = [f'tasmaxbinF_{k}' for k in ds.binF.values]
                ds = ds.rename({'binF':'pred'})
            elif params_proc[proc]['varname'] == 'days90plus':
                ds = ds.rename({'binF':'pred'})
            else:
                raise KeyError()

            # Rename proj_bases to allow for cross-processing
            #ds = ds.rename({'proj_base':'proj_base_bcds'})
        
            # Calculate impact from linear regression model
            ds[proc] = (coeffs.coeffs*ds[params_proc[proc]['varname']]).sum('pred',skipna=False)
            # Calculate change in impact vs. GWL0.61
            ds['d'+proc] = ds[proc] - ds[proc].sel(gwl=0.61)

            # Add max gwl to coordinates, for referece
            ds = ds.assign_coords({'maxgwl':(('idv'),ds.gwl.isel(gwl=np.array([(g-1) if g != 0 else -1 
                                                              for g in (np.isnan(ds['d'+proc]).
                                                                        argmax('gwl').
                                                                        max([d for d in ds['d'+proc].sizes if d not in ['gwl','idv']])).
                                                                        load()])).values)})
            # Make sure all-nan idvs are all-nan in maxgwl
            ds['maxgwl'] = ds.maxgwl.where(~np.isnan(ds['d'+proc]).all([d for d in ds['d'+proc].sizes if d != 'idv']).load())
        
            # Prepare output 
            ds_out = ds[['locid','d'+proc]]
            ds_out.attrs = {'SOURCE':'project_impacts.ipynb',
                    'DESCRIPTION':f'changes in {proc}, at the {params_proc[proc]['geo_typ']} level, projecting models fit on each of four reanalyses using each of four reanalysis-based bias-corrected and downscaled climate states',
                    'pop_weighted':params_proc[proc]['popwght']}
        
            # Export
            with warnings.catch_warnings():
                # Catches a Zarr v3 error related to string coordinates saved 
                # as objects instead of strings, I believe
                warnings.filterwarnings('ignore')
                utility_save(ds_out,output_fn,zarr_mode='w',save_kwargs = {'zarr_format':3,'consolidated':False})
        else:
            print(output_fn+' exists, skipped!')

  0%|          | 0/12 [00:00<?, ?it/s]

/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/mortality/dmortality-crossfit_agg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_impact-regions.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/mortality/dmortality-crossfit_agg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_impact-regions.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/mortality/dmortality-crossfit_agg_CESM2-WACCM_ALLRUNS_ALLGWLs_impact-regions.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/mortality/dmortality-crossfit_agg_CESM2-WACCM_ALLRUNS_ALLGWLs_impact-regions.zarr saved!


...

/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_MIROC6_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_MIROC6_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_MPI-ESM1-2-LR_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_MPI-ESM1-2-LR_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_MRI-ESM2-0_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_MRI-ESM2-0_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


In [6]:
ds = xr.open_zarr('/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/gdppc/dgdppc-crossfit_agg_IPSL-CM6A-LR_ALLRUNS_ALLGWLs_adm1_popwght.zarr')

/glade/derecho/scratch/schwarzwald/tmp/ipykernel_18158/3633411549.py:1: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds = xr.open_zarr('/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/gdppc/dgdppc-crossfit_agg_IPSL-CM6A-LR_ALLRUNS_ALLGWLs_adm1_popwght.zarr')
/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


In [13]:
ds.sel(proj_base='MERRA2').mean('poly_idx').sel(gwl=2).dgdppc.values

array([[-4.10945955, -3.36918912, -3.25394402, -3.48210533, -3.93427686,
        -3.99239193, -4.2070303 , -3.53140631, -3.83588581, -3.8697678 ,
        -3.64179368, -3.80173396, -3.51293996, -4.01804743, -4.76994623,
        -3.99452357, -3.93795716, -4.29664621, 25.74045847, -4.29209103,
        -3.70318648, -4.04652488, -3.3264241 , -4.38584807, -4.28828325,
        -3.73488266, -3.74242133, -4.02019488],
       [-1.15998685, -0.5879026 , -0.54496768, -0.66336801, -1.06818973,
        -1.23048756, -1.39786154, -0.62024778, -0.97209931, -0.96877854,
        -0.92856007, -0.90429483, -0.69637661, -1.02622556, -1.90246065,
        -1.08453618, -1.30946704, -1.37171101, 23.46858258, -1.38940237,
        -0.91013054, -1.08116958, -0.54675008, -1.40344876, -1.43314388,
        -0.63865756, -0.92761742, -1.24854609],
       [-4.93560356, -4.13629993, -4.00045917, -4.26113756, -4.73567404,
        -4.76939852, -5.0007823 , -4.33445264, -4.63439515, -4.67827449,
        -4.39850268, -4.6077